In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

P = Path("../Data/Processed_Datasets")
ship = pd.read_csv(P / "shipments_full.csv", parse_dates=["OrderDate"])
lines_all = pd.read_csv(P / "order_lines_all.csv", parse_dates=["OrderDate"])
print(ship.shape, lines_all.shape)

(5000, 10) (15000, 11)


In [2]:
# Order-level features
order_feats = (lines_all.groupby("OrderID")
    .agg(n_lines=("OrderItemID","size"), total_qty=("Quantity","sum"),
         order_value=("LineValue","sum"), n_categories=("Category","nunique"),
         max_qty=("Quantity","max")).reset_index())

df = ship.merge(order_feats, on="OrderID", how="left", validate="one_to_one")
num_cols = ["n_lines","total_qty","order_value","n_categories","max_qty"]
df["has_items"] = df["n_lines"].notna().astype(int)
df[num_cols] = df[num_cols].fillna(0)

# Calendar and location features
df["order_month"] = df["OrderDate"].dt.month
df["order_dow"] = df["OrderDate"].dt.dayofweek
df["cross_city"] = (df["WarehouseCity"] != df["CustomerCity"]).astype(int)

# Warehouse load: shipments from the same warehouse in the previous 7 days
df = df.sort_values("OrderDate", kind="stable").reset_index(drop=True)
def load7(g):
    d = g["OrderDate"].values
    hi = np.searchsorted(d, d, side="left")
    lo = np.searchsorted(d, d - np.timedelta64(7, "D"), side="left")
    return pd.Series(hi - lo, index=g.index)
df["wh_load_7d"] = df.groupby("WarehouseID", group_keys=False).apply(load7)

# Target and split (time-based: 70% train, 15% validation, 15% test)
df["is_delayed"] = (df["DelayDays"] >= 3).astype(int)
n = len(df)
df["split"] = "train"
df.loc[int(0.70*n):, "split"] = "val"
df.loc[int(0.85*n):, "split"] = "test"

print(df.groupby("split").agg(rows=("is_delayed","size"),
      start=("OrderDate","min"), end=("OrderDate","max"),
      delay_rate=("is_delayed","mean")).round(3))
print(df[["wh_load_7d","cross_city","order_value"]].describe().round(2))

df.to_csv(P / "delay_features.csv", index=False)

C:\Users\USER\AppData\Local\Temp\ipykernel_14796\690016729.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["wh_load_7d"] = df.groupby("WarehouseID", group_keys=False).apply(load7)


       rows      start        end  delay_rate
split                                        
test    750 2026-08-03 2026-09-08       0.655
train  3500 2026-01-01 2026-06-25       0.631
val     750 2026-06-25 2026-08-03       0.648
       wh_load_7d  cross_city  order_value
count      5000.0     5000.00      5000.00
mean          9.2        0.75      4333.61
std           3.2        0.43      3221.50
min           0.0        0.00         0.00
25%           7.0        1.00      1915.01
50%           9.0        1.00      3764.40
75%          11.0        1.00      6117.97
max          20.0        1.00     19290.34


In [3]:
RAW = Path("../Data/Original_Datasets")
prod = pd.read_csv(RAW / "SwiftLog_Products.csv")
inv_clean = pd.read_csv(P / "inventory_clean.csv")
lc = pd.read_csv(P / "order_lines_clean.csv", parse_dates=["OrderDate"])

lc["month"] = lc["OrderDate"].dt.month
lc = lc[lc["month"] <= 8]

monthly = (lc.groupby(["ProductID","month"])
             .agg(units=("Quantity","sum"), orders=("OrderID","nunique"))
             .reset_index())

grid = pd.MultiIndex.from_product([prod["ProductID"], range(1, 9)],
                                  names=["ProductID","month"]).to_frame(index=False)
dem = (grid.merge(monthly, on=["ProductID","month"], how="left")
           .fillna({"units": 0, "orders": 0})
           .sort_values(["ProductID","month"]).reset_index(drop=True))

g = dem.groupby("ProductID")
dem["lag_1"] = g["units"].shift(1)
dem["lag_2"] = g["units"].shift(2)
dem["lag_mean"] = dem[["lag_1","lag_2"]].mean(axis=1)
dem["orders_lag_1"] = g["orders"].shift(1)

dem = dem.merge(prod, on="ProductID", how="left", validate="many_to_one")
stock = (inv_clean.groupby("ProductID")
         .agg(total_stock=("Stock","sum"), n_warehouses=("WarehouseID","nunique"))
         .reset_index())
dem = dem.merge(stock, on="ProductID", how="left")
dem[["total_stock","n_warehouses"]] = dem[["total_stock","n_warehouses"]].fillna(0)

dem = dem.dropna(subset=["lag_2"]).copy()
dem["split"] = np.where(dem["month"] <= 6, "train", np.where(dem["month"] == 7, "val", "test"))

print(dem.groupby("split").agg(rows=("units","size"), mean_units=("units","mean"),
                               std_units=("units","std")).round(1))
print("corr(lag_1, units):", round(dem["lag_1"].corr(dem["units"]), 3))
print("corr(total_stock, units):", round(dem["total_stock"].corr(dem["units"]), 3))
print("products with zero stock rows:", (dem["total_stock"] == 0).sum())

dem.to_csv(P / "demand_features.csv", index=False)

       rows  mean_units  std_units
split                             
test    250        31.2       14.4
train  1000        27.6       13.8
val     250        23.3       12.4
corr(lag_1, units): 0.008
corr(total_stock, units): 0.046
products with zero stock rows: 0


In [4]:
cust = pd.read_csv(RAW / "SwiftLog_Customers.csv")
orders = pd.read_csv(RAW / "SwiftLog_Orders.csv")
orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], format="%m/%d/%Y")

CUT = pd.Timestamp("2026-06-01")   # history before this, target after
END = pd.Timestamp("2026-09-01")
HIST_DAYS = (CUT - pd.Timestamp("2026-01-01")).days   # 151

oh = (orders[orders["OrderDate"] < CUT].groupby("CustomerID")
      .agg(n_orders_all=("OrderID","size"),
           n_cancelled=("Status", lambda s: (s == "Cancelled").sum()),
           first_order=("OrderDate","min"), last_order=("OrderDate","max")))
oh["cancel_rate"] = oh["n_cancelled"] / oh["n_orders_all"]

la = lines_all[lines_all["Status"] != "Cancelled"]
hist = la[la["OrderDate"] < CUT]
lh = (hist.groupby("CustomerID")
      .agg(n_orders=("OrderID","nunique"), n_lines=("OrderItemID","size"),
           total_qty=("Quantity","sum"), total_spend=("LineValue","sum"),
           n_categories=("Category","nunique")))
lh["avg_order_value"] = lh["total_spend"] / lh["n_orders"]

share = hist.pivot_table(index="CustomerID", columns="Category", values="LineValue",
                         aggfunc="sum", fill_value=0)
share = share.div(share.sum(axis=1), axis=0).add_prefix("share_")

# Target: spend in the future window
fut = (la[(la["OrderDate"] >= CUT) & (la["OrderDate"] < END)]
       .groupby("CustomerID")["LineValue"].sum().rename("future_spend"))

cv = (cust.drop(columns="CustomerName")
          .merge(oh, on="CustomerID", how="left")
          .merge(lh, on="CustomerID", how="left")
          .merge(share, on="CustomerID", how="left")
          .merge(fut, on="CustomerID", how="left"))

cv["recency_days"] = (CUT - cv["last_order"]).dt.days.fillna(HIST_DAYS)
cv["tenure_days"] = (CUT - cv["first_order"]).dt.days.fillna(0)
cv = cv.drop(columns=["first_order","last_order"])
zero_cols = [c for c in cv.columns if c not in ("CustomerID","City","recency_days","tenure_days")]
cv[zero_cols] = cv[zero_cols].fillna(0)

rng = np.random.RandomState(42)
cv["split"] = rng.choice(["train","val","test"], size=len(cv), p=[0.70, 0.15, 0.15])

print(cv["split"].value_counts().to_dict())
print(cv["future_spend"].describe().round(0))
print("customers with zero future spend:", (cv["future_spend"] == 0).mean().round(3))
print("corr(total_spend, future_spend):", round(cv["total_spend"].corr(cv["future_spend"]), 3))
print("corr(n_orders, future_spend):", round(cv["n_orders"].corr(cv["future_spend"]), 3))

cv.to_csv(P / "customer_features.csv", index=False)

{'train': 712, 'test': 153, 'val': 135}
count     1000.0
mean      5345.0
std       5982.0
min          0.0
25%          0.0
50%       3551.0
75%       8614.0
max      38800.0
Name: future_spend, dtype: float64
customers with zero future spend: 0.295
corr(total_spend, future_spend): -0.003
corr(n_orders, future_spend): 0.031


In [5]:
w_raw = pd.read_csv(RAW / "SwiftLog_Warehouses.csv")
dfd = pd.read_csv(P / "delay_features.csv", parse_dates=["OrderDate"])
dfd["month"] = dfd["OrderDate"].dt.month
dfd = dfd[dfd["month"] <= 8]

wm = (dfd.groupby(["WarehouseID","month"])
        .agg(n_shipments=("ShipmentID","size"), delay_rate=("is_delayed","mean"),
             avg_delay_days=("DelayDays","mean"), avg_order_value=("order_value","mean"))
        .reset_index().sort_values(["WarehouseID","month"]).reset_index(drop=True))

g = wm.groupby("WarehouseID")
for c in ["delay_rate","avg_delay_days","n_shipments","avg_order_value"]:
    wm[f"lag_{c}"] = g[c].shift(1)

stock = (inv_clean.groupby("WarehouseID")
         .agg(total_stock=("Stock","sum"), n_products=("ProductID","nunique"),
              avg_stock=("Stock","mean"),
              n_low_stock=("Stock", lambda s: (s < 50).sum())).reset_index())
wm = (wm.merge(w_raw, on="WarehouseID", how="left")
        .merge(stock, on="WarehouseID", how="left")
        .rename(columns={"Location":"WarehouseCity"}))

wm = wm.dropna(subset=["lag_delay_rate"]).copy()
wm["split"] = np.where(wm["month"] <= 5, "train", np.where(wm["month"] == 6, "val", "test"))

print(wm.shape)
print(wm.groupby("split").agg(rows=("delay_rate","size"), mean=("delay_rate","mean"),
                              std=("delay_rate","std")).round(3))
print("avg shipments per warehouse-month:", round(wm["n_shipments"].mean(), 1))
print("corr(lag_delay_rate, delay_rate):", round(wm["lag_delay_rate"].corr(wm["delay_rate"]), 3))
print("spread between warehouses (std of warehouse means):",
      round(wm.groupby("WarehouseID")["delay_rate"].mean().std(), 3))

wm.to_csv(P / "warehouse_features.csv", index=False)

df = pd.read_csv(P / "delay_features.csv")
df.to_csv(P.parent / "Feature_Engineered_Dataset.csv", index=False)

(105, 16)
       rows   mean    std
split                    
test     30  0.643  0.076
train    60  0.634  0.069
val      15  0.630  0.098
avg shipments per warehouse-month: 40.5
corr(lag_delay_rate, delay_rate): -0.106
spread between warehouses (std of warehouse means): 0.03
